In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
df_cancer = pd.read_csv("cancer de mama.csv")
df_diabetes = pd.read_excel("diabetes.xlsx")

print("Cáncer de mama (filas, columnas):", df_cancer.shape)
print("Diabetes (filas, columnas):", df_diabetes.shape)


In [ ]:
print("--- COLUMNAS CÁNCER DE MAMA ---")
print(df_cancer.columns.tolist())
df_cancer.info()
display(df_cancer.head())
display(df_cancer.describe())

In [ ]:
print("\n--- COLUMNAS DIABETES ---")
print(df_diabetes.columns.tolist())
df_diabetes.info()
display(df_diabetes.head())
display(df_diabetes.describe())

In [ ]:
print("\nNulos en diabetes:\n", df_diabetes.isnull().sum())


In [ ]:
df_cancer = pd.read_csv("cancer de mama.csv", sep=";")
print("Nulos en cáncer de mama:\n", df_cancer.isnull().sum())

In [ ]:
df_cancer_32 = df_cancer.copy()
df_diabetes_32 = df_diabetes.copy()

# ---------------------------------------------------------
# 1. ELIMINACIÓN DE DUPLICADOS (si los hay)
# ---------------------------------------------------------
print("Duplicados antes - Cáncer:", df_cancer_32.duplicated().sum())
print("Duplicados antes - Diabetes:", df_diabetes_32.duplicated().sum())

df_cancer_32 = df_cancer_32.drop_duplicates()
df_diabetes_32 = df_diabetes_32.drop_duplicates()

In [ ]:
# 2. CORRECCIÓN DE TIPOS DE DATOS INCORRECTOS
# ---------------------------------------------------------
# En cáncer de mama, si alguna columna numérica quedó como texto (object) por comas o espacios,
# la convertimos a numérico (forzando errores a NaN si aparecieran)
for col in df_cancer_32.columns:
  if col not in ["id", "diagnosis"]:
    df_cancer_32[col] = pd.to_numeric(df_cancer_32[col], errors="coerce")

# Aseguramos que las columnas numéricas de diabetes sean numéricas
cols_num_diab = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]
for col in cols_num_diab:
  df_diabetes_32[col] = pd.to_numeric(df_diabetes_32[col], errors="coerce")

In [ ]:
3. #TRATAMIENTO DE VALORES NULOS (y ceros inválidos en diabetes)
# ---------------------------------------------------------
# En diabetes, los ceros en estas variables representan datos faltantes (no es posible tener 0 en glucosa, IMC, etc.)
cols_cero_faltante = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]
df_diabetes_32[cols_cero_faltante] = df_diabetes_32[cols_cero_faltante].replace(
    0, np.nan
)

# Imputamos valores nulos numéricos con la MEDIANA (robusta frente a valores atípicos/outliers)
for col in df_cancer_32.select_dtypes(include=[np.number]).columns:
  if col != "id" and df_cancer_32[col].isnull().sum() > 0:
    df_cancer_32[col] = df_cancer_32[col].fillna(df_cancer_32[col].median())

for col in cols_num_diab:
  if col != "Outcome" and df_diabetes_32[col].isnull().sum() > 0:
    df_diabetes_32[col] = df_diabetes_32[col].fillna(
        df_diabetes_32[col].median()
    )

# Si existiera algún nulo en 'Outcome' o 'diagnosis', se rellena con la MODA
if df_cancer_32["diagnosis"].isnull().sum() > 0:
  df_cancer_32["diagnosis"] = df_cancer_32["diagnosis"].fillna(
      df_cancer_32["diagnosis"].mode()[0]
  )

if df_diabetes_32["Outcome"].isnull().sum() > 0:
  df_diabetes_32["Outcome"] = df_diabetes_32["Outcome"].fillna(
      df_diabetes_32["Outcome"].mode()[0]
  )

In [ ]:
# 4. NORMALIZACIÓN O ESTANDARIZACIÓN DE TEXTO
# ---------------------------------------------------------
# Limpiamos espacios sobrantes y estandarizamos mayúsculas en 'diagnosis'
df_cancer_32["diagnosis"] = (
    df_cancer_32["diagnosis"].astype(str).str.strip().str.upper()
)

# Verificación final de nulos
print("\nNulos finales - Cáncer:", df_cancer_32.isnull().sum().sum())
print("Nulos finales - Diabetes:", df_diabetes_32.isnull().sum().sum())

In [ ]:
print("========== CÁNCER DE MAMA (DESPUÉS DE 3.2) ==========")
print("Dimensiones (filas, columnas):", df_cancer_32.shape)
print("Duplicados restantes:", df_cancer_32.duplicated().sum())
print("Nulos por columna:\n", df_cancer_32.isnull().sum())
print("\nTipos de datos:\n", df_cancer_32.dtypes)
print("\nValores únicos en diagnosis:", df_cancer_32["diagnosis"].unique())
display(df_cancer_32.head())

In [ ]:
print("\n========== DIABETES (DESPUÉS DE 3.2) ==========")
print("Dimensiones (filas, columnas):", df_diabetes_32.shape)
print("Duplicados restantes:", df_diabetes_32.duplicated().sum())
print("Nulos por columna:\n", df_diabetes_32.isnull().sum())
print("\nCeros restantes en columnas clínicas clave:")
cols_clinicas = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print((df_diabetes_32[cols_clinicas] == 0).sum())
print("\nTipos de datos:\n", df_diabetes_32.dtypes)
display(df_diabetes_32.head())
display(df_diabetes_32.describe())

In [ ]:
# Partimos de los DataFrames limpios del paso 3.2
df_cancer_33 = df_cancer_32.copy()
df_diabetes_33 = df_diabetes_32.copy()

# ---------------------------------------------------------
# 1. ELIMINACIÓN DE COLUMNAS IRRELEVANTES O REDUNDANTES
# ---------------------------------------------------------
# En cáncer de mama:
# - 'id' es solo un identificador único sin valor clínico ni estadístico.
# - Las columnas '_se' y '_worst' son redundantes si nuestro análisis principal se centra
#   en el promedio ('_mean'); las eliminamos para reducir dimensionalidad y ruido.
cols_cancer_eliminar = ["id"] + [
    col
    for col in df_cancer_33.columns
    if col.endswith("_se") or col.endswith("_worst")
]
df_cancer_33 = df_cancer_33.drop(columns=cols_cancer_eliminar)

# En diabetes:
# Todas las columnas (Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin,
# BMI, DiabetesPedigreeFunction, Age, Outcome) son variables clínicas o de resultado
# relevantes para el análisis, por lo que conservamos todas.

# ---------------------------------------------------------
# 2. CREACIÓN DE NUEVAS VARIABLES DERIVADAS
# ---------------------------------------------------------
# En cáncer de mama:
# - Diagnostico_Label: etiqueta legible ("Maligno" / "Benigno") para gráficas y reportes.
# - Ratio_Area_Radio: relación entre área y radio promedio para comparar compacidad/forma.
df_cancer_33["Diagnostico_Label"] = df_cancer_33["diagnosis"].map(
    {"M": "Maligno", "B": "Benigno"}
)
df_cancer_33["Ratio_Area_Radio"] = (
    df_cancer_33["area_mean"] / df_cancer_33["radius_mean"]
)

# En diabetes:
# - Resultado_Label: etiqueta legible ("Sin diabetes" / "Con diabetes").
# - Categoria_BMI: clasificación clínica estándar del Índice de Masa Corporal.
# - Grupo_Edad: segmentación etaria para comparar grupos de pacientes.
df_diabetes_33["Resultado_Label"] = df_diabetes_33["Outcome"].map(
    {0: "Sin diabetes", 1: "Con diabetes"}
)
df_diabetes_33["Categoria_BMI"] = pd.cut(
    df_diabetes_33["BMI"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Bajo peso", "Normal", "Sobrepeso", "Obesidad"],
)
df_diabetes_33["Grupo_Edad"] = pd.cut(
    df_diabetes_33["Age"],
    bins=[0, 29, 44, 59, 120],
    labels=["Joven (<=29)", "Adulto joven (30-44)", "Adulto medio (45-59)", "Mayor (60+)"],
)

print("Columnas finales - Cáncer:", df_cancer_33.columns.tolist())
print("Columnas finales - Diabetes:", df_diabetes_33.columns.tolist())

In [ ]:
# --- 1. ESTADÍSTICAS DESCRIPTIVAS: CÁNCER DE MAMA ---
num_cancer = df_cancer_33.select_dtypes(include=[np.number]).columns

resumen_cancer = pd.DataFrame({
    "Media": df_cancer_33[num_cancer].mean(),
    "Mediana": df_cancer_33[num_cancer].median(),
    "Moda": df_cancer_33[num_cancer].mode().iloc[0],
    "Desv_Estandar": df_cancer_33[num_cancer].std(),
})

print("=== CÁNCER DE MAMA: MEDIA, MEDIANA, MODA Y DESV. ESTÁNDAR ===")
display(resumen_cancer.round(2))

print("\n=== CÁNCER DE MAMA: TABLA DESCRIBE() ===")
display(df_cancer_33[num_cancer].describe().round(2))

frec_cancer = pd.DataFrame({
    "Frecuencia": df_cancer_33["Diagnostico_Label"].value_counts(),
    "Proporcion_%": (
        df_cancer_33["Diagnostico_Label"].value_counts(normalize=True) * 100
    ).round(2),
})
print("\n=== CÁNCER DE MAMA: FRECUENCIAS Y PROPORCIONES ===")
display(frec_cancer)

# --- 2. ESTADÍSTICAS DESCRIPTIVAS: DIABETES ---
num_diabetes = df_diabetes_33.select_dtypes(include=[np.number]).columns

resumen_diabetes = pd.DataFrame({
    "Media": df_diabetes_33[num_diabetes].mean(),
    "Mediana": df_diabetes_33[num_diabetes].median(),
    "Moda": df_diabetes_33[num_diabetes].mode().iloc[0],
    "Desv_Estandar": df_diabetes_33[num_diabetes].std(),
})

print("\n=== DIABETES: MEDIA, MEDIANA, MODA Y DESV. ESTÁNDAR ===")
display(resumen_diabetes.round(2))

print("\n=== DIABETES: TABLA DESCRIBE() ===")
display(df_diabetes_33[num_diabetes].describe().round(2))

frec_diabetes = pd.DataFrame({
    "Frecuencia": df_diabetes_33["Resultado_Label"].value_counts(),
    "Proporcion_%": (
        df_diabetes_33["Resultado_Label"].value_counts(normalize=True) * 100
    ).round(2),
})
print("\n=== DIABETES: FRECUENCIAS Y PROPORCIONES ===")
display(frec_diabetes)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
conteos = df[col_outcome].value_counts().sort_index()
etiquetas = [f"Clase {k}" for k in conteos.index]
if set(conteos.index) == {0, 1}:
  etiquetas = ["Sin diabetes (0)", "Con diabetes (1)"]

barras = ax.bar(etiquetas, conteos.values, color=palette_diag, width=0.5)
for b in barras:
  h = b.get_height()
  pct = h / len(df) * 100
  ax.text(
      b.get_x() + b.get_width() / 2,
      h + len(df) * 0.015,
      f"{h} ({pct:.1f}%)",
      ha="center",
      va="bottom",
      fontweight="bold",
  )
ax.set_title(
    "Distribución de casos según diagnóstico", fontsize=13, fontweight="bold"
)
ax.set_ylabel("Número de registros")
plt.tight_layout()
plt.savefig("grafica_1_diagnostico.png", dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for val, lbl, col in zip(
    sorted(df[col_outcome].unique()), etiquetas, palette_diag
):
  subset = df[df[col_outcome] == val][col_glucose].dropna()
  ax.hist(
      subset, bins=20, alpha=0.65, label=lbl, color=col, edgecolor="white"
  )
ax.set_title(
    f"Distribución de {col_glucose} por diagnóstico",
    fontsize=13,
    fontweight="bold",
)
ax.set_xlabel(col_glucose)
ax.set_ylabel("Frecuencia")
ax.legend()
plt.tight_layout()
plt.savefig("grafica_2_glucosa.png", dpi=300)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=df, x=col_outcome, y=col_bmi, palette=palette_diag, ax=ax)
ax.set_title(
    f"Comparación de {col_bmi} según diagnóstico", fontsize=13, fontweight="bold"
)
ax.set_xlabel("Diagnóstico")
ax.set_ylabel(col_bmi)
plt.tight_layout()
plt.savefig("grafica_3_imc_boxplot.png", dpi=300)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Filtrar glucosa > 0 para evitar el ruido de ceros inválidos
df_clean = df[df[col_glucose] > 0]

sns.scatterplot(
    data=df_clean,
    x=col_age,
    y=col_glucose,
    hue=col_outcome,
    palette=palette_diag,
    alpha=0.75,
    s=45,
    edgecolor="white",
    linewidth=0.4,
    ax=ax,
)

ax.set_title(
    "Glucosa vs. Edad por diagnóstico", fontsize=13, fontweight="bold", pad=12
)
ax.set_xlabel("Edad (años)")
ax.set_ylabel("Glucosa (mg/dL)")
ax.legend(title="Diagnóstico", labels=["Sin diabetes (0)", "Con diabetes (1)"])

plt.tight_layout()
plt.savefig("grafica_4_dispersion.png", dpi=300)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title(
    "Matriz de correlación entre variables numéricas",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("grafica_5_correlacion.png", dpi=300)
plt.show()